# ResNet-UNet

This notebook uses the shared preprocessing notebook before defining the model.

In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


from torchvision.models import resnet34, ResNet34_Weights

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")
print(f"Using torch version: {torch.__version__}")

Using device: cpu
Using torch version: 2.13.0


In [5]:
# running preprocessing notebook

%run ../shared/preprocessing.ipynb

In [6]:
# Creating Data Loaders

train_loader, val_loader, test_loader, pos_weight = get_dataloaders(
    data_dir=DEFAULT_DATA_DIR,
    batch_size=32,
    augment_train=True,
    num_workers=0,
)

print("Input channels:", NUM_CHANNELS)
print("Positive-class weight:", pos_weight)
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

I0000 00:00:1789529413.404130  219594 tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144


Input channels: 12
Positive-class weight: 90.01595545900618
Train batches: 469
Validation batches: 59
Test batches: 53


In [7]:
inputs, targets, valid_mask = next(iter(train_loader))

print("Inputs: ", inputs.shape, inputs.dtype)
print("Targets: ", targets.shape, targets.dtype)
print("Valid mask: ", valid_mask.shape, valid_mask.dtype)

print("Target values: ", torch.unique(targets))
print("Mask values: ", torch.unique(valid_mask))

assert inputs.shape[1:] == (12, 64, 64)
assert targets.shape[1:] == (1, 64, 64)
assert valid_mask.shape == targets.shape
assert torch.isfinite(inputs).all()

print("Data contract verified successfully.")

Inputs:  torch.Size([32, 12, 64, 64]) torch.float32
Targets:  torch.Size([32, 1, 64, 64]) torch.float32
Valid mask:  torch.Size([32, 1, 64, 64]) torch.float32
Target values:  tensor([-1.,  0.,  1.])
Mask values:  tensor([0., 1.])
Data contract verified successfully.
